In [ ]:
import json
import logging
import os
import sqlite3

from datetime import datetime, timedelta
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("OPENAI_API_KEY is not set.")

DB_PATH = Path.cwd() / "db" / "workouts.db"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

log = logging.getLogger("workout_logger")

def get_db_connection() -> sqlite3.Connection:
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row
    return conn

def init_db() -> None:
    with get_db_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS workouts (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                exercise    TEXT    NOT NULL,
                sets        INTEGER,
                reps        INTEGER,
                weight_kg   REAL,
                notes       TEXT,
                logged_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        conn.commit()

init_db()

print(f"Database initialized at {DB_PATH}.")

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "log_workout",
            "description": (
                "Log a completed workout session. Use this when the user says they "
                "did, completed, or finished an exercise. Extract exercise name, sets, "
                "reps, and weight from the message."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "exercise": {
                        "type": "string",
                        "description": "Name of the exercise, e.g. 'bench press', 'squat', 'deadlift'"
                    },
                    "sets": {
                        "type": "integer",
                        "description": "Number of sets performed"
                    },
                    "reps": {
                        "type": "integer",
                        "description": "Number of reps per set"
                    },
                    "weight_kg": {
                        "type": "number",
                        "description": "Weight used in kilograms. Use null if bodyweight exercise."
                    },
                    "notes": {
                        "type": "string",
                        "description": "Optional notes, e.g. 'felt heavy', 'PR attempt'"
                    }
                },
                "required": ["exercise"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_workout_history",
            "description": (
                "Retrieve past workout sessions. Use when the user asks about their "
                "history, what they did recently, or how many times they trained a specific exercise."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "exercise": {
                        "type": "string",
                        "description": "Filter by exercise name. Omit to get all workouts."
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Maximum number of records to return. Default is 10."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_personal_record",
            "description": (
                "Get the personal record (heaviest weight) for a specific exercise."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "exercise": {
                        "type": "string",
                        "description": "The exercise to look up the PR for."
                    }
                },
                "required": ["exercise"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weekly_summary",
            "description": (
                "Get a summary of workouts for a specific week. "
                "Use when the user asks about 'this week', 'last week', or 'weekly stats'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "weeks_ago": {
                        "type": "integer",
                        "description": "0 = current week, 1 = last week, etc. Default is 0."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "delete_last_workout",
            "description": (
                "Delete the most recently logged workout. Use only when the user "
                "explicitly asks to undo, delete, or remove their last entry."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

In [ ]:
def log_workout(
    exercise: str,
    sets: int | None = None,
    reps: int | None = None,
    weight_kg: float | None = None,
    notes: str | None = None,
) -> str:
    with get_db_connection() as conn:
        conn.execute(
            """INSERT INTO workouts (exercise, sets, reps, weight_kg, notes)
               VALUES (?, ?, ?, ?, ?)""",
            (exercise.lower().strip(), sets, reps, weight_kg, notes),
        )
        
        conn.commit()

    parts = [f"Logged: {exercise}"]

    if sets and reps:
        parts.append(f"{sets}x{reps}")
    if weight_kg:
        parts.append(f"at {weight_kg}kg")
    if notes:
        parts.append(f"({notes})")

    return " ".join(parts)

In [ ]:
def get_workout_history(exercise: str | None = None, limit: int = 10) -> str:
    limit = max(1, min(limit, 100))

    with get_db_connection() as conn:
        if exercise:
            rows = conn.execute(
                """SELECT exercise, sets, reps, weight_kg, notes, logged_at
                   FROM workouts WHERE exercise LIKE ?
                   ORDER BY logged_at DESC LIMIT ?""",
                (f"%{exercise.lower()}%", limit),
            ).fetchall()
        else:
            rows = conn.execute(
                """SELECT exercise, sets, reps, weight_kg, notes, logged_at
                   FROM workouts ORDER BY logged_at DESC LIMIT ?""",
                (limit,),
            ).fetchall()

    if not rows:
        return "No workout history found."

    return json.dumps([dict(row) for row in rows], default=str)

In [ ]:
def get_personal_record(exercise: str) -> str:
    with get_db_connection() as conn:
        row = conn.execute(
            """SELECT weight_kg, sets, reps, logged_at
               FROM workouts
               WHERE exercise LIKE ? AND weight_kg IS NOT NULL
               ORDER BY weight_kg DESC LIMIT 1""",
            (f"%{exercise.lower()}%",),
        ).fetchone()

    if not row:
        return f"No records found for '{exercise}'."

    return json.dumps(dict(row), default=str)

In [ ]:
def get_weekly_summary(weeks_ago: int = 0) -> str:
    weeks_ago = max(0, min(weeks_ago, 52))

    today = datetime.now().date()
    start_of_week = today - timedelta(days=today.weekday())
    week_start = start_of_week - timedelta(weeks=weeks_ago)
    week_end = week_start + timedelta(days=6)

    with get_db_connection() as conn:
        rows = conn.execute(
            """SELECT exercise, sets, reps, weight_kg
               FROM workouts WHERE DATE(logged_at) BETWEEN ? AND ?""",
            (week_start.isoformat(), week_end.isoformat()),
        ).fetchall()

    if not rows:
        label = "This week" if weeks_ago == 0 else f"{weeks_ago} week(s) ago"
        return f"{label}: no workouts logged."

    sessions = len(rows)
    total_sets = sum(r['sets'] or 0 for r in rows)

    total_volume = sum(
        (r['sets'] or 0) * (r['reps'] or 0) * (r['weight_kg'] or 0)
        for r in rows
    )

    exercises = list(set(r['exercise'] for r in rows))

    label = "This week" if weeks_ago == 0 else f"{weeks_ago} week(s) ago"
    
    return json.dumps({
        "period": f"{week_start} to {week_end}",
        "label": label,
        "sessions": sessions,
        "total_sets": total_sets,
        "total_volume_kg": round(total_volume, 1),
        "exercises": exercises,
    })

In [ ]:
def delete_last_workout() -> str:
    with get_db_connection() as conn:
        row = conn.execute(
            "SELECT id, exercise, logged_at FROM workouts ORDER BY logged_at DESC LIMIT 1"
        ).fetchone()

        if not row:
            return "No workouts to delete."

        conn.execute("DELETE FROM workouts WHERE id = ?", (row['id'],))
        conn.commit()
        
        return f"Deleted: {row['exercise']} logged at {row['logged_at']}"

In [ ]:
from openai import OpenAI

client = OpenAI()

MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

TOOL_FUNCTIONS = {
    "log_workout": log_workout,
    "get_workout_history": get_workout_history,
    "get_personal_record": get_personal_record,
    "get_weekly_summary": get_weekly_summary,
    "delete_last_workout": delete_last_workout,
}

SYSTEM_PROMPT = """
You are a helpful workout tracking assistant.
You help users log their workouts and query their training history.
Always confirm when you've logged something. Be concise and friendly.
When displaying data, format it in a readable way for the user.
"""

In [ ]:
def run_agent_stream(user_message: str, conversation_history: list):
    conversation_history.append({"role": "user", "content": user_message})

    while True:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                *conversation_history,
            ],
            tools=TOOLS,
            tool_choice="auto",
            stream=True,
        )

        collected_content = ""
        tool_calls_by_index: dict[int, dict] = {}

        for chunk in stream:
            delta = chunk.choices[0].delta

            if delta.content:
                collected_content += delta.content
                yield collected_content

            if delta.tool_calls:
                for tc_delta in delta.tool_calls:
                    idx = tc_delta.index

                    if idx not in tool_calls_by_index:
                        tool_calls_by_index[idx] = {
                            "id": "",
                            "function": {"name": "", "arguments": ""},
                        }

                    entry = tool_calls_by_index[idx]

                    if tc_delta.id:
                        entry["id"] = tc_delta.id

                    if tc_delta.function:
                        if tc_delta.function.name:
                            entry["function"]["name"] += tc_delta.function.name
                        if tc_delta.function.arguments:
                            entry["function"]["arguments"] += tc_delta.function.arguments

        assistant_msg = {"role": "assistant", "content": collected_content or None}

        if tool_calls_by_index:
            assistant_msg["tool_calls"] = [
                {"id": tc["id"], "type": "function", "function": tc["function"]}
                for _, tc in sorted(tool_calls_by_index.items())
            ]

        conversation_history.append(assistant_msg)

        if not tool_calls_by_index:
            return

        for _, tc in sorted(tool_calls_by_index.items()):
            function_name = tc["function"]["name"]

            try:
                function_args = json.loads(tc["function"]["arguments"])
            except json.JSONDecodeError as exc:
                log.error("Bad arguments from model for %s: %s", function_name, exc)
                result = f"Error: could not parse arguments — {exc}"

                conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": result,
                })

                continue

            log.info("Tool call: %s(%s)", function_name, function_args)

            if function_name not in TOOL_FUNCTIONS:
                result = f"Error: unknown tool '{function_name}'"
            else:
                try:
                    result = TOOL_FUNCTIONS[function_name](**function_args)
                except Exception as exc:
                    log.exception("Tool %s raised an error", function_name)
                    result = f"Error executing {function_name}: {exc}"

            conversation_history.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": str(result),
            })

In [ ]:
import gradio as gr

def _build_display(openai_history: list) -> list[dict]:
    return [
        {"role": msg["role"], "content": msg["content"]}
        for msg in openai_history
        if msg.get("role") in ("user", "assistant") and msg.get("content")
    ]

def add_user_message(
    user_message: str, chatbot_display: list, openai_history: list
) -> tuple[str, list, list]:
    if not user_message.strip():
        return "", chatbot_display, openai_history

    chatbot_display = chatbot_display + [{"role": "user", "content": user_message}]

    return "", chatbot_display, openai_history

def bot_respond(chatbot_display: list, openai_history: list):
    user_message = chatbot_display[-1]["content"]

    partial = ""

    for partial in run_agent_stream(user_message, openai_history):
        yield (
            chatbot_display + [{"role": "assistant", "content": partial}],
            openai_history,
        )

    yield _build_display(openai_history), openai_history

with gr.Blocks(title="Workout Logger") as demo:
    gr.Markdown(
        "## Workout Logger\n"
        "Log your workouts and track your progress through conversation."
    )

    chatbot = gr.Chatbot(label="Chat", height=500, type="messages")

    with gr.Row():
        msg_input = gr.Textbox(
            placeholder="e.g. 'I just did 4x8 bench press at 80kg'",
            label="Message",
            scale=5,
            show_label=False,
        )

        send_btn = gr.Button("Send", scale=1, variant="primary")

    gr.Examples(
        examples=[
            "Log 4 sets of squat, 5 reps at 100kg",
            "What's my deadlift PR?",
            "Show me my bench press history",
            "How did this week look?",
            "Delete my last entry",
        ],
        inputs=msg_input,
    )

    state = gr.State([])

    send_btn.click(
        fn=add_user_message,
        inputs=[msg_input, chatbot, state],
        outputs=[msg_input, chatbot, state],
    ).then(
        fn=bot_respond,
        inputs=[chatbot, state],
        outputs=[chatbot, state],
    )

    msg_input.submit(
        fn=add_user_message,
        inputs=[msg_input, chatbot, state],
        outputs=[msg_input, chatbot, state],
    ).then(
        fn=bot_respond,
        inputs=[chatbot, state],
        outputs=[chatbot, state],
    )

In [ ]:
demo.launch()